The goal is to:
- Learn patterns between resumes and job descriptions
- Predict candidate suitability
- Compare multiple models and select the best one


In [39]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import joblib
import os

In [40]:
pip install xgboost

In [41]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

In [42]:
X_train = sp.load_npz("../data/X_train.npz")
X_test  = sp.load_npz("../data/X_test.npz")
y_train = np.load("../data/y_train.npy")
y_test  = np.load("../data/y_test.npy")
print("✅ Data loaded!")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

✅ Data loaded!
X_train: (6240, 10001)
X_test: (1759, 10001)


In [43]:
## Training Models

In [44]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        solver='lbfgs'   # ✅ supports multiclass automatically
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        random_state=42,
        use_label_encoder=False,
        eval_metric='mlogloss'
    )
}

The models are trained on:
- TF-IDF vectors of combined resume and job description text
- These vectors represent important keywords in numerical form

In [45]:
results = {}

for name, model in models.items():
    print(f"\n🔹 Training {name}...")

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='weighted')

    results[name] = {
        'model': model,
        'accuracy': acc,
        'f1': f1,
        'y_pred': y_pred
    }

    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")


🔹 Training Logistic Regression...


Accuracy: 0.5196
F1 Score: 0.4837

🔹 Training Random Forest...
Accuracy: 0.5355
F1 Score: 0.4414

🔹 Training XGBoost...


c:\Users\diyaa\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [17:40:56] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Accuracy: 0.5185
F1 Score: 0.4899


In [46]:
print(results.keys())

dict_keys(['Logistic Regression', 'Random Forest', 'XGBoost'])


We trained three different machine learning models on the TF-IDF features extracted from the cleaned dataset:

- Logistic Regression  
- Random Forest  
- XGBoost  

### Training Results

| Model                 | Accuracy | F1 Score |
|----------------------|---------:|---------:|
| Logistic Regression  | 0.5196   | 0.4837   |
| Random Forest        | 0.5355   | 0.4414   |
| XGBoost              | 0.5185   | 0.4899   |

### Model Selection

Although Random Forest achieved the highest accuracy, we selected **XGBoost** as the best model because it achieved the highest **F1 Score (0.4899)**.

F1 Score is preferred over accuracy because:
- It balances precision and recall
- It is more reliable for imbalanced datasets
- It ensures better performance across all classes

In [47]:
import joblib
import os

# create folder
os.makedirs("../models", exist_ok=True)

# select best model (you already know XGBoost is best)
best_model = results["XGBoost"]['model']

# save it
joblib.dump(best_model, "../models/best_model.pkl")

print("✅ Best model saved!")

✅ Best model saved!
